# 23_descendant_languages: What Survives Time?

**Lab report:** Invariant Specification Demystifies Translation

Notebook 17 examined script change:

```text
same language
different scripts
```

Notebook 23 examines language evolution:

```text
same specification
different descendant languages
```

This is the strongest transformation tested so far.

Core question:

> What survives centuries to millennia of accumulated linguistic change?

## 1. Setup

Clone or reuse the upstream GLOSSOPETRAE repository.

In [ ]:
from pathlib import Path
import os
import re
import json
import csv
import subprocess
from collections import Counter, defaultdict
from itertools import combinations

ROOT = Path.cwd()
UPSTREAM_URL = "https://github.com/elder-plinius/GLOSSOPETRAE.git"
UPSTREAM_DIR = ROOT / "GLOSSOPETRAE"

if not UPSTREAM_DIR.exists():
    subprocess.run(["git", "clone", UPSTREAM_URL, str(UPSTREAM_DIR)], check=True)
else:
    print("Upstream repo already present:", UPSTREAM_DIR)

FIGURES = ROOT / "figures"
DATA = ROOT / "data"
FIGURES.mkdir(exist_ok=True)
DATA.mkdir(exist_ok=True)

print("Repository exists:", UPSTREAM_DIR.exists())

## 2. Locate evolution and lineage artifacts

Search for the parts of the upstream repo that model:

- ancestors,
- descendants,
- daughter languages,
- sound change,
- lineage,
- language-family generation.

In [ ]:
evolution_terms = [
    "evolution",
    "evolve",
    "descendant",
    "daughter",
    "lineage",
    "ancestor",
    "proto",
    "family",
    "branch",
    "drift",
    "sound change",
    "sound_change",
    "phonological change",
    "lexical change",
    "semantic change",
    "grammar change",
    "mutation",
    "generation",
]

TEXT_SUFFIXES = {
    ".py", ".md", ".txt", ".yaml", ".yml", ".json", ".toml",
    ".js", ".ts", ".html", ".css", ".csv", ".tsv"
}

path_hits = []
content_hits = []

for path in UPSTREAM_DIR.rglob("*"):
    if ".git" in path.parts or path.is_dir():
        continue

    rel = path.relative_to(UPSTREAM_DIR)
    lower_path = str(rel).lower()

    hits = [t for t in evolution_terms if t in lower_path]
    if hits:
        path_hits.append({
            "path": str(rel),
            "suffix": path.suffix,
            "hits": ", ".join(hits),
            "size_bytes": path.stat().st_size if path.exists() else None,
        })

    if path.suffix.lower() not in TEXT_SUFFIXES:
        continue

    try:
        text = path.read_text(errors="ignore")
    except Exception:
        continue

    lower = text.lower()
    for term in evolution_terms:
        count = lower.count(term)
        if count:
            content_hits.append({
                "path": str(rel),
                "term": term,
                "count": count,
            })

len(path_hits), len(content_hits)

In [ ]:
try:
    import pandas as pd

    path_df = pd.DataFrame(path_hits).sort_values(["hits", "path"])
    content_df = pd.DataFrame(content_hits)

    display(path_df.head(100))

    term_summary = (
        content_df
        .groupby("term", as_index=False)["count"]
        .sum()
        .sort_values("count", ascending=False)
    )
    display(term_summary)

    path_df.to_csv(DATA / "23_evolution_path_hits.csv", index=False)
    term_summary.to_csv(DATA / "23_evolution_term_summary.csv", index=False)
except Exception:
    print("Path hits:")
    for row in path_hits[:100]:
        print(row)
    print("\\nTerm counts:")
    print(Counter([r["term"] for r in content_hits]))

## 3. Extract evolution snippets

Look near high-value words to understand whether evolution is modeled as:

1. explicit rules,
2. stochastic drift,
3. sound-change tables,
4. lineage trees,
5. repeated generation,
6. post-hoc naming.

In [ ]:
def snippets_for(term, max_snippets=20, radius=200):
    rows = []
    for path in UPSTREAM_DIR.rglob("*"):
        if ".git" in path.parts or path.is_dir() or path.suffix.lower() not in TEXT_SUFFIXES:
            continue

        try:
            text = path.read_text(errors="ignore")
        except Exception:
            continue

        lower = text.lower()
        start_search = 0
        while len(rows) < max_snippets:
            idx = lower.find(term.lower(), start_search)
            if idx < 0:
                break

            start = max(0, idx - radius)
            end = min(len(text), idx + len(term) + radius)
            snippet = text[start:end].replace("\\n", " ")
            rows.append({
                "term": term,
                "path": str(path.relative_to(UPSTREAM_DIR)),
                "snippet": snippet,
            })
            start_search = idx + len(term)

        if len(rows) >= max_snippets:
            break

    return rows

snippet_terms = ["evolution", "descendant", "daughter", "lineage", "ancestor", "proto", "sound change", "drift"]
snippets = []
for term in snippet_terms:
    snippets.extend(snippets_for(term, max_snippets=12))

try:
    import pandas as pd
    snippets_df = pd.DataFrame(snippets)
    display(snippets_df)
    snippets_df.to_csv(DATA / "23_evolution_snippets.csv", index=False)
except Exception:
    for s in snippets:
        print("\\n---", s["term"], s["path"])
        print(s["snippet"])

## 4. Candidate family-generation files

Rank likely files that implement or document descendant-language generation.

In [ ]:
candidate_evolution_files = []

for path in UPSTREAM_DIR.rglob("*"):
    if ".git" in path.parts or path.is_dir() or path.suffix.lower() not in TEXT_SUFFIXES:
        continue

    rel = path.relative_to(UPSTREAM_DIR)
    lower = str(rel).lower()

    path_score = sum(1 for term in evolution_terms if term in lower)

    try:
        text = path.read_text(errors="ignore")
    except Exception:
        text = ""

    content_score = sum(text.lower().count(term) for term in evolution_terms)

    if path_score or content_score:
        candidate_evolution_files.append({
            "path": str(rel),
            "suffix": path.suffix,
            "size_bytes": path.stat().st_size,
            "path_score": path_score,
            "content_score": content_score,
            "total_score": path_score * 10 + content_score,
        })

candidate_evolution_files = sorted(
    candidate_evolution_files,
    key=lambda r: (-r["total_score"], r["size_bytes"], r["path"])
)

try:
    import pandas as pd
    evo_files_df = pd.DataFrame(candidate_evolution_files)
    display(evo_files_df.head(100))
    evo_files_df.to_csv(DATA / "23_candidate_evolution_files.csv", index=False)
except Exception:
    for row in candidate_evolution_files[:100]:
        print(row)

## 5. Descendant-language invariant table

This is the strongest invariant test so far.

Unlike script change, language evolution may change sound, words, grammar, meaning, and writing systems.

Some invariants may fail. That is useful information.

In [ ]:
descendant_invariant_rows = [
    {
        "property": "lineage",
        "expected_under_evolution": "preserved",
        "observed": "unresolved",
        "notes": "Descendant languages should preserve ancestry metadata or recoverable family structure."
    },
    {
        "property": "correspondence",
        "expected_under_evolution": "partially preserved",
        "observed": "unresolved",
        "notes": "Cognates and mapped concepts may persist even when forms change."
    },
    {
        "property": "meaning / concept",
        "expected_under_evolution": "partially preserved",
        "observed": "unresolved",
        "notes": "Some meanings should remain stable; semantic drift may occur."
    },
    {
        "property": "lexicon / words",
        "expected_under_evolution": "changes",
        "observed": "unresolved",
        "notes": "Forms may shift, split, merge, or be replaced."
    },
    {
        "property": "phonology / pronunciation",
        "expected_under_evolution": "changes systematically",
        "observed": "unresolved",
        "notes": "Sound shifts are expected to be structured rather than random."
    },
    {
        "property": "grammar",
        "expected_under_evolution": "may change",
        "observed": "unresolved",
        "notes": "Grammar may drift while preserving some structural relations."
    },
    {
        "property": "script / glyphs",
        "expected_under_evolution": "may change or be independent",
        "observed": "unresolved",
        "notes": "Writing systems may evolve separately from spoken language."
    },
    {
        "property": "specification constraints",
        "expected_under_evolution": "partially preserved",
        "observed": "unresolved",
        "notes": "The strongest version of the lab claim depends on this."
    },
]

try:
    import pandas as pd
    descendant_df = pd.DataFrame(descendant_invariant_rows)
    display(descendant_df)
    descendant_df.to_csv(DATA / "23_descendant_invariant_template.csv", index=False)
except Exception:
    for row in descendant_invariant_rows:
        print(row)

## 6. Family tree model

This figure expresses the expected structure:

```text
Specification
      ↓
Proto Language
      ↓
Evolution Rules
      ↓
A, B, C
```

Descendants are not identical, but they may remain related by lineage and correspondence.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 6))
ax.axis("off")

nodes = {
    "Specification\\n(constraints)": (0.50, 0.92),
    "Proto Language\\n(ancestor)": (0.50, 0.74),
    "Evolution Rules\\n(sound, lexicon, grammar, meaning)": (0.50, 0.56),
    "Descendant A": (0.22, 0.34),
    "Descendant B": (0.50, 0.34),
    "Descendant C": (0.78, 0.34),
    "Lineage + Correspondence\\n(invariant candidates)": (0.50, 0.12),
}

for label, (x, y) in nodes.items():
    ax.text(
        x, y, label,
        ha="center", va="center",
        bbox=dict(boxstyle="round,pad=0.45", linewidth=1.2),
        fontsize=10
    )

edges = [
    ("Specification\\n(constraints)", "Proto Language\\n(ancestor)"),
    ("Proto Language\\n(ancestor)", "Evolution Rules\\n(sound, lexicon, grammar, meaning)"),
    ("Evolution Rules\\n(sound, lexicon, grammar, meaning)", "Descendant A"),
    ("Evolution Rules\\n(sound, lexicon, grammar, meaning)", "Descendant B"),
    ("Evolution Rules\\n(sound, lexicon, grammar, meaning)", "Descendant C"),
    ("Descendant A", "Lineage + Correspondence\\n(invariant candidates)"),
    ("Descendant B", "Lineage + Correspondence\\n(invariant candidates)"),
    ("Descendant C", "Lineage + Correspondence\\n(invariant candidates)"),
]

for a, b in edges:
    x1, y1 = nodes[a]
    x2, y2 = nodes[b]
    ax.annotate(
        "",
        xy=(x2, y2 + 0.04),
        xytext=(x1, y1 - 0.04),
        arrowprops=dict(arrowstyle="->", linewidth=1)
    )

# Pairwise relatedness among descendants
for x1, x2 in [(0.22, 0.50), (0.50, 0.78)]:
    ax.annotate(
        "",
        xy=(x2 - 0.08, 0.34),
        xytext=(x1 + 0.08, 0.34),
        arrowprops=dict(arrowstyle="<->", linewidth=1.1, linestyle="--")
    )

ax.text(0.36, 0.39, "family relation", ha="center", fontsize=8)
ax.text(0.64, 0.39, "family relation", ha="center", fontsize=8)

path = FIGURES / "23_specification_lineage.png"
plt.savefig(path, dpi=180, bbox_inches="tight")
print("Saved:", path)
plt.show()

## 7. Family similarity scaffold

If the upstream engine emits word lists, concept tables, or descendant metadata, compute simple pairwise similarities.

This scaffold is deliberately generic. Replace `family_records` with actual outputs when available.

In [ ]:
def jaccard(a, b):
    a = set(a)
    b = set(b)
    if not a and not b:
        return None
    return len(a & b) / len(a | b)

# TODO: replace these placeholders with generated outputs.
family_records = {
    # "proto": {"forms": [...], "concepts": [...], "phonemes": [...]},
    # "descendant_a": {"forms": [...], "concepts": [...], "phonemes": [...]},
    # "descendant_b": {"forms": [...], "concepts": [...], "phonemes": [...]},
}

similarity_rows = []

for left, right in combinations(family_records.keys(), 2):
    lrec = family_records[left]
    rrec = family_records[right]
    for field in ["forms", "concepts", "phonemes"]:
        similarity_rows.append({
            "left": left,
            "right": right,
            "field": field,
            "jaccard": jaccard(lrec.get(field, []), rrec.get(field, [])),
        })

try:
    import pandas as pd
    sim_df = pd.DataFrame(similarity_rows)
    display(sim_df)
    sim_df.to_csv(DATA / "23_family_similarity_template.csv", index=False)
except Exception:
    for row in similarity_rows:
        print(row)

if not family_records:
    print("family_records is empty. Fill it with upstream descendant outputs.")

## 8. Evolution-event extraction scaffold

If outputs include explicit change events, collect them into a table.

Target schema:

| generation | branch | change_type | before | after | rule | evidence_path |
|---|---|---|---|---|---|---|

In [ ]:
change_event_terms = {
    "sound": ["sound", "phoneme", "phonology", "pronunciation"],
    "lexical": ["lexicon", "word", "root", "vocabulary"],
    "semantic": ["semantic", "meaning", "concept"],
    "grammar": ["grammar", "syntax", "morphology"],
    "script": ["script", "glyph", "orthography"],
}

change_events = []

for path in UPSTREAM_DIR.rglob("*"):
    if ".git" in path.parts or path.is_dir() or path.suffix.lower() not in TEXT_SUFFIXES:
        continue

    try:
        text = path.read_text(errors="ignore")
    except Exception:
        continue

    lower = text.lower()
    if not any(term in lower for terms in change_event_terms.values() for term in terms):
        continue

    for change_type, terms in change_event_terms.items():
        if any(term in lower for term in terms) and any(t in lower for t in ["change", "evolve", "drift", "shift", "descendant"]):
            change_events.append({
                "evidence_path": str(path.relative_to(UPSTREAM_DIR)),
                "change_type": change_type,
                "note": "candidate file mentions change/evolution language",
            })

try:
    import pandas as pd
    events_df = pd.DataFrame(change_events).drop_duplicates()
    display(events_df.head(100))
    events_df.to_csv(DATA / "23_candidate_change_events.csv", index=False)
except Exception:
    for event in change_events[:100]:
        print(event)

## 9. Work area: generate a language family

After locating the smallest upstream command, run it here.

Target output:

```text
Proto Language
   ├─ Descendant A
   ├─ Descendant B
   └─ Descendant C
```

Record:

1. ancestor specification,
2. evolution rules,
3. descendant outputs,
4. correspondence metadata,
5. visible changes,
6. persistent lineage.

In [ ]:
# TODO: replace with the smallest upstream language-family command.
#
# Example shape:
#
# result = subprocess.run(
#     ["python", "..."],
#     cwd=UPSTREAM_DIR,
#     text=True,
#     capture_output=True,
#     check=True,
# )
# print(result.stdout[:2000])
#
# Then parse or summarize:
#
# proto_language = ...
# descendants = ...
# evolution_rules = ...
# correspondence = ...

## 10. Hypothesis 4

**Hypothesis 4**

Language evolution changes observable forms.

Sounds change.

Words change.

Grammar and meaning may drift.

Scripts may change independently.

Yet descendant languages can retain correspondence relationships inherited from a common specification.

Lineage may be a stronger invariant than individual words, sounds, or glyphs.

## 11. Questions for Notebook 29

Notebook 29 should unify the three transformations:

```text
translation
script change
language evolution
```

Question:

> What invariant explains all three?

Candidate answers:

- specification,
- meaning,
- correspondence,
- lineage,
- constraint structure.

Notebook 29 should score these candidates across every transformation studied so far.